In [1]:
import csv
from text_index import TextIndexer
from inverted_index import InvertedIndex, InvertedFile, BUCKET_LIMIT


CSV_PATH = "spotify_millsongdata.csv"
INV_PATH = "inv.dat"
DOC_PATH = "inv_doc.dat"

In [3]:
print(BUCKET_LIMIT)

65536


In [4]:
indexer = TextIndexer(INV_PATH, DOC_PATH)

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)   # respeta comillas y saltos de línea
    for i, row in enumerate(reader):
        doc_id = f"d{i}"                # o cualquier id único
        lyrics = (row.get("text") or "").strip()
        if not lyrics:
            continue
        indexer.add_document(doc_id, lyrics)

indexer.finalize()


In [ ]:

def main():
    f = InvertedFile(INV_PATH)
    n = f._read_header()
    print(f"[INFO] num_buckets={n}, BUCKET_LIMIT={BUCKET_LIMIT}")

    for i in range(min(n, 3)):
        b = f.read(i)
        print(f"\nBucket {i}:")
        print(b)
        import pickle
        print("[DEBUG] tamaño (bytes):", len(pickle.dumps(b)))

if __name__ == "__main__":
    main()


[INFO] num_buckets=1867, BUCKET_LIMIT=65536

Bucket 0:
{'look': {'d0': 2, 'd2': 1, 'd3': 1, 'd4': 1, 'd5': 1, 'd11': 3, 'd13': 1, 'd17': 2, 'd22': 1, 'd28': 1, 'd33': 2, 'd34': 4, 'd35': 2}, 'face': {'d0': 2, 'd12': 2, 'd13': 1, 'd14': 1, 'd21': 2, 'd25': 2}, 'wonder': {'d0': 1, 'd12': 1, 'd21': 1}, 'mean': {'d0': 1, 'd3': 1, 'd4': 1, 'd12': 1, 'd13': 1, 'd14': 3, 'd29': 3, 'd33': 2}, 'someth': {'d0': 1, 'd8': 1, 'd13': 1, 'd18': 2}, 'special': {'d0': 1, 'd34': 1}, 'way': {'d0': 1, 'd2': 2, 'd5': 1, 'd7': 2, 'd8': 1, 'd10': 1, 'd13': 2, 'd14': 2, 'd20': 1, 'd21': 2, 'd28': 6, 'd29': 3, 'd30': 2, 'd31': 1, 'd33': 4, 'd34': 2}, 'smile': {'d0': 1, 'd3': 1, 'd4': 1, 'd8': 1, 'd12': 1, 'd13': 2, 'd15': 2, 'd34': 3}, 'see': {'d0': 1, 'd6': 1, 'd7': 4, 'd8': 2, 'd11': 3, 'd12': 2, 'd13': 1, 'd14': 2, 'd15': 3, 'd18': 1, 'd21': 2, 'd22': 2, 'd24': 1, 'd27': 2, 'd31': 1, 'd32': 1, 'd33': 1, 'd34': 4}, 'lucki': {'d0': 1}, 'one': {'d0': 1, 'd5': 1, 'd6': 2, 'd7': 1, 'd8': 1, 'd10': 2, 'd13': 1, '

In [6]:


def main():
    f = InvertedFile(INV_PATH)
    n_before = f._read_header()
    print(f"[INFO] buckets antes de build_index: {n_before}, BUCKET_LIMIT={BUCKET_LIMIT}")

    inv = InvertedIndex(INV_PATH, docfile_path=DOC_PATH)
    inv.build_index()

    inv.compute_tfidf_norms()

    f2 = InvertedFile(INV_PATH)
    n_after = f2._read_header()
    print(f"[INFO] buckets después de build_index: {n_after}")

    # Opcional: ver los primeros 2 buckets
    for i in range(min(n_after, 10)):
        b = f2.read(i)
        print(f"\nBucket {i}:")
        print(b)

if __name__ == "__main__":
    main()


[INFO] buckets antes de build_index: 1867, BUCKET_LIMIT=65536
[DBG] round=1, n=1867, B=128, fanin=127
[DBG] round=2, n=672, B=128, fanin=127
[DBG] round=3, n=668, B=128, fanin=127
[DBG] round=4, n=667, B=128, fanin=127
[DBG] round=5, n=667, B=128, fanin=127
[INFO] buckets después de build_index: 667

Bucket 0:
{'0': {'d415': 1, 'd416': 1, 'd1117': 25, 'd3804': 2, 'd5215': 3, 'd7285': 1, 'd7697': 3, 'd8541': 1, 'd10118': 1, 'd10533': 1, 'd11554': 1}, '00': {'d3172': 1, 'd4566': 1, 'd8542': 1, 'd8794': 1, 'd11581': 1}, '000': {'d306': 1, 'd589': 1, 'd625': 1, 'd961': 7, 'd1231': 2, 'd2667': 6, 'd2725': 1, 'd4011': 1, 'd5215': 3, 'd5504': 1, 'd6651': 3, 'd7292': 1, 'd8407': 1, 'd8415': 3, 'd8546': 1, 'd10082': 1, 'd11571': 1, 'd12070': 1}, '007': {'d11071': 1}, '009': {'d8272': 1}, '00a': {'d2061': 1}, '01': {'d5215': 1, 'd6460': 1, 'd9179': 1}, '03': {'d8553': 1, 'd11701': 1}, '04': {'d11525': 1}, '06': {'d5786': 1, 'd8851': 1}, '07': {'d5786': 1}, '0h': {'d1091': 1}, '0o': {'d4702': 3},

In [7]:
def load_mapping(csv_path: str = CSV_PATH):
    mapping = {}
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            did = f"d{i}"    # MISMO esquema que en el indexer
            mapping[did] = {
                "artist": row["artist"],
                "song": row["song"],
                "link": row["link"],
                "text": row["text"],
            }
    return mapping

mapping = load_mapping()

print("Índice y mapping listos ✅")


Índice y mapping listos ✅


In [8]:

inv = InvertedIndex(INV_PATH, docfile_path=DOC_PATH)

def buscar(q: str, k: int = 5):
    q = q.strip()
    if not q:
        print("[WARN] query vacía")
        return

    results = inv.search(q, limit=k)
    if not results:
        print(f"[RESULT] '{q}' -> sin resultados")
        return

    print(f"[RESULT] '{q}' ->")
    for rank, (doc_id, score) in enumerate(results, start=1):
        info = mapping.get(doc_id, {})
        artist = info.get("artist", "?")
        song = info.get("song", "?")
        print(f"  {rank}. {artist} – {song}  (doc_id={doc_id}, score={score:.4f})")


In [11]:
buscar("look at her face")

[RESULT] 'look at her face' ->
  1. Foreigner – Face To Face  (doc_id=d32751, score=0.7412)
  2. Quiet Riot – Face To Face  (doc_id=d49602, score=0.6383)
  3. Pet Shop Boys – A Face Like That  (doc_id=d15793, score=0.6030)
  4. Michael Bolton – Take A Look At My Face  (doc_id=d43615, score=0.5155)
  5. Waylon Jennings – Cindy, Oh Cindy  (doc_id=d21145, score=0.5012)


In [12]:
buscar("losing")

[RESULT] 'losing' ->
  1. Britney Spears – He About To Lose Me  (doc_id=d26462, score=0.9145)
  2. Queen – Don't Lose Your Head  (doc_id=d49297, score=0.8547)
  3. Bruno Mars – Lost  (doc_id=d26650, score=0.7900)
  4. Emmylou Harris – Get Up John  (doc_id=d31472, score=0.7718)
  5. Roy Orbison – Losing You  (doc_id=d18368, score=0.7535)
